**Reference Link:** [RAG Systems Essentials (Analytics Vidhya)](https://courses.analyticsvidhya.com/courses/take/rag-systems-essentials/lessons/60148017-hands-on-deep-dive-into-rag-evaluation-metrics-generator-metrics-i)

# Build a Simple RAG System

# Build a Simple RAG System - Steps Overview

## 1. Setup and Dependencies
- Install OpenAI and LangChain dependencies
- Run setup notebook for environment configuration

## 2. OpenAI Embedding Models
- Initialize OpenAI embedding model (`text-embedding-3-small`)
- Demonstrate proper usage of `embed_query()` vs `embed_documents()`
- Show embedding vector dimensions (1536 for text-embedding-3-small)

## 3. Document Loading and Processing
- **JSON Documents**: Load Wikipedia data using `JSONLoader`
- **PDF Documents**: Load multiple PDF files using `PyMuPDFLoader`
- **Document Chunking**: Create simple chunks with RecursiveCharacterTextSplitter
  - Chunk size: 3500 characters
  - Chunk overlap: 200 characters
- **Data Processing**: Combine Wikipedia and PDF documents into unified format

## 4. Vector Database Setup
- Initialize Chroma vector database
- Index document chunks with embeddings
- Configure cosine similarity distance function
- Save database to disk for persistence
- Demonstrate loading from disk

## 5. Retrieval System

- Create similarity-based retriever
- Configure to return top 5 most similar documents
- Test retrieval with sample queries

## 6. RAG Pipeline Construction
- **Prompt Template**: Create RAG prompt for question-answering
- **Chain Assembly**: Build RAG chain using:
  - Document retriever
  - Document formatter
  - Prompt template
  - LLM (GPT-4o-mini)
- **Pipeline Flow**: Context retrieval → Document formatting → Prompt generation → LLM response

## 7. Testing and Validation
- Test RAG system with various queries:
  - Machine learning concepts
  - Computer vision topics (CNN, Vision Transformers)
  - NLP and linguistics
  - Self-attention mechanisms
  - AI system types
  - Framework comparisons

## 8. Alternative Processing Strategies
- Provide batch processing options for large datasets
- Include rate limiting and retry logic
- Suggest local embedding alternatives for API limit issues

In [ ]:
from pprint import pprint
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import JSONLoader
import json
from langchain.docstore.document import Document
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from glob import glob
from IPython.display import display, Markdown
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from rich import print #

### Open AI Embedding Models

LangChain enables us to access Open AI embedding models which include the newest models: a smaller and highly efficient `text-embedding-3-small` model, and a larger and more powerful `text-embedding-3-large` model.

In [ ]:
# details here: https://openai.com/blog/new-embedding-models-and-api-updates
openai_embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
text = "Hello, world!"
embedding = openai_embed_model.embed_documents(text)
print("Length of embedding:", len(embedding))
print("Length of embedding[0]:", len(embedding[0]))

### Why is the length 13?
- The string "Hello, world!" contains exactly 13 characters, including the comma, space, and exclamation mark:
- H-e-l-l-o-,-[space]-w-o-r-l-d-!
- Therefore, you get a length of 13—one embedding vector is created for each character.

### The Correct Way
Here are the proper ways to use OpenAI embeddings:

For a single document:


In [ ]:
# Option 1: Use embed_query for single text
embedding = openai_embed_model.embed_query("Hello, world!")
# print("embedding:", embedding)
print(f"Length: {len(embedding)}")  # This will be 1536 (for text-embedding-3-small)

# Option 2: Use embed_documents with a list
embedding = openai_embed_model.embed_documents(["Hello, world!"])
# print("embedding:", embedding)
print(f"Number of embeddings: {len(embedding)}")  # This will be 1 (one embedding vector)
print(f"Each embedding dimension: {len(embedding[0])}")  # This will be 1536

For multiple documents:


In [ ]:
texts = ["Hello, world!", "How are you?", "This is a test."]
embeddings_list = openai_embed_model.embed_documents(texts)
print(f"Number of embeddings: {len(embeddings_list)}")  # 3
print(f"Each embedding dimension: {len(embeddings_list[0])}")  # 1536

What You're Actually Seeing
The actual embedding vector dimensions for OpenAI's `text-embedding-3-small` model should be 1536, not 13. The 13 you're seeing is the number of character-level embeddings created by the incorrect usage.
Try using embeddings.embed_query("Hello, world!") instead, and you'll see the proper embedding vector with 1536 dimensions!

### Load and Process JSON Documents

In [ ]:

loader = JSONLoader(file_path='../../data/wikidata_rag_demo.jsonl',
                    jq_schema='.',
                    text_content=False,
                    json_lines=True)

wiki_docs = loader.load()

print("Document 0 Contents:", wiki_docs[0].page_content)
print("Document 0 metadata:", wiki_docs[0].metadata)
print("Document 3 Contents:", wiki_docs[3].page_content)
print("Length of wiki_docs:", len(wiki_docs))

In [ ]:

wiki_docs_processed = []

for doc in wiki_docs:
    metadata = doc.metadata
    doc = json.loads(doc.page_content)
    metadata = {
        "title": doc['title'],
        "id": doc['id'],
        "source": metadata["source"]
    }
    data = ' '.join(doc['paragraphs'])
    wiki_docs_processed.append(Document(page_content=data, metadata=metadata))

print("wiki_docs_processed[3]:", wiki_docs_processed[3])

### Load and Process PDF documents

#### Create Simple Document Chunks for Standard Retrieval

Here we just use simple chunking where each chunk is a fixed size of <= 3500 characters and overlap of 200 characters for any small isolated chunks (you can and should experiment with various chunk sizes and overlaps)

In [ ]:


def create_simple_chunks(file_path, chunk_size=3500, chunk_overlap=0):

    print('Loading pages:', file_path)
    loader = PyMuPDFLoader(file_path)
    doc_pages = loader.load()

    print('Chunking pages:', file_path)
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,
                                              chunk_overlap=chunk_overlap)
    doc_chunks = splitter.split_documents(doc_pages)
    print('Finished processing:', file_path)
    print()
    return doc_chunks

In [ ]:


pdf_files = glob('../../data/*.pdf')

In [ ]:
paper_docs = []
for fp in pdf_files:
    paper_docs.extend(create_simple_chunks(file_path=fp,
                                           chunk_size=3500,
                                           chunk_overlap=200))

In [ ]:
len(paper_docs)

### Combine all document chunks in one list

In [ ]:
print("Length of paper_docs:", len(paper_docs))
print("Length of wiki_docs_processed:", len(wiki_docs_processed))
print("Type of wiki_docs_processed:", type(wiki_docs_processed))
print("Type of paper_docs:", type(paper_docs))
print("wiki_docs_processed[3]:", wiki_docs_processed[3])
print("paper_docs[0]:", paper_docs[0])

In [ ]:
total_docs = wiki_docs_processed + paper_docs
print("Length of total_docs:", len(total_docs))

## Index Document Chunks and Embeddings in Vector DB

Here we initialize a connection to a Chroma vector DB client, and also we want to save to disk, so we simply initialize the Chroma client and pass the directory where we want the data to be saved to.

In [ ]:


# create vector DB of docs and embeddings - takes < 30s on Colab
chroma_db = Chroma.from_documents(documents=total_docs,
                                  collection_name='my_db',
                                  embedding=openai_embed_model,
                                  # need to set the distance function to cosine else it uses euclidean by default
                                  # check https://docs.trychroma.com/guides#changing-the-distance-function
                                  collection_metadata={"hnsw:space": "cosine"},
                                  persist_directory="./my_db")

## Alternative Batch Processing Strategies

If you're still encountering issues with large datasets, here are some additional strategies:

### 1. Further Reduce Batch Size
If you still get errors, try reducing the batch size even more:

```python
# For very large datasets or API rate limits
chroma_db = create_chroma_db_in_batches(
    documents=total_docs,
    batch_size=1000,  # Even smaller batches
    collection_name='my_db',
    embedding_function=openai_embed_model,
    persist_directory="./my_db"
)
```

### 2. Add Rate Limiting and Retry Logic
For production use, consider adding rate limiting:

```python
import time
from langchain_chroma import Chroma

def create_chroma_db_with_rate_limit(documents, batch_size=2000, delay=1):
    """Create Chroma DB with rate limiting between batches"""
    
    if len(documents) > 0:
        # Process first batch
        first_batch = documents[:batch_size]
        chroma_db = Chroma.from_documents(
            documents=first_batch,
            collection_name='my_db',
            embedding=openai_embed_model,
            collection_metadata={"hnsw:space": "cosine"},
            persist_directory="./my_db"
        )
        
        # Process remaining batches with delays
        for i in range(batch_size, len(documents), batch_size):
            batch = documents[i:i + batch_size]
            print(f"Processing batch {i//batch_size + 1}, sleeping {delay}s...")
            time.sleep(delay)  # Rate limiting
            chroma_db.add_documents(batch)
            
        return chroma_db
```

### 3. Use Local Embeddings Instead
If OpenAI API limits are problematic, consider using local embeddings:

```python
from langchain_huggingface import HuggingFaceEmbeddings

# Use a local embedding model (no API limits)
local_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# This won't have API batch size limits
chroma_db = Chroma.from_documents(
    documents=total_docs,
    collection_name='my_db',
    embedding=local_embeddings,
    collection_metadata={"hnsw:space": "cosine"},
    persist_directory="./my_db"
)
```


### Load Vector DB from disk

This is just to show once you have a vector database on disk you can just load and create a connection to it anytime

In [ ]:
# load from disk
chroma_db = Chroma(persist_directory="./my_db",
                   collection_name='my_db',
                   embedding_function=openai_embed_model)

### Semantic Similarity based Retrieval

We use simple cosine similarity here and retrieve the top 5 similar documents based on the user input query

In [ ]:
similarity_retriever = chroma_db.as_retriever(search_type="similarity",
                                              search_kwargs={"k": 5})

In [ ]:


def display_docs(docs):
    for doc in docs:
        print('Metadata:', doc.metadata)
        print('Content Brief:')
        display(Markdown(doc.page_content[:1000]))
        print()

In [ ]:
query = "what is machine learning?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is the difference between transformers and vision transformers?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

## Build the RAG Pipeline

In [ ]:


rag_prompt = """You are an assistant who is an expert in question-answering tasks.
                Answer the following question using only the following pieces of retrieved context.
                If the answer is not in the context, do not make up answers, just say that you don't know.
                Keep the answer detailed and well formatted based on the information from the context.

                Question:
                {question}

                Context:
                {context}

                Answer:
            """

rag_prompt_template = ChatPromptTemplate.from_template(rag_prompt)

In [ ]:

chatgpt = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_rag_chain = (
    {
        "context": (similarity_retriever
                      |
                    format_docs),
        "question": RunnablePassthrough()
    }
      |
    rag_prompt_template
      |
    chatgpt
)

In [ ]:
query = "What is machine learning?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

In [ ]:
query = "What is a CNN?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

In [ ]:
query = "What is NLP and its relation to linguistics?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

In [ ]:
query = "How is self-attention important in transformers?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

In [ ]:
query = "What is an Agentic AI System?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

In [ ]:
query = "What is LangChain?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

In [ ]:
query = "What is LangGraph?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

In [ ]:
query = "What is the difference between transformers and vision transformers?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

In [ ]:
query = "How is a resnet better than a CNN?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))